<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/Supertonic-3_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Supertonic 3 — Lightning-Fast, On-Device, Multilingual TTS

99M-parameter open-weight TTS from Supertone, running natively via ONNX Runtime. 31 languages, 10 built-in voices, 44.1 kHz high-quality audio, expression tags for natural prosody, and **no GPU required** — runs entirely on CPU.

### Quick Start
1. Connect to any runtime — **CPU is fine, GPU not needed**
2. Run Steps 1–4 in order — no token, no sign-up needed
3. Open the Gradio UI link from Step 4 and start generating

| Runtime | Notes |
|---------|-------|
| CPU (default) | Fastest cold start, lowest VRAM, no quota limits |
| T4 GPU | Works but offers no real speedup (ONNX Runtime is CPU-optimized here) |
| L4 / A100 | No speedup vs CPU — not worth paying for |

**First run downloads ~400 MB** (ONNX models + 10 voice styles) to `~/.cache/supertonic3/`.

**Expression tags** can be dropped into your text for natural human nuance: `<laugh>`, `<breath>`, `<sigh>`, and 7 more. No prompt engineering, no reference audio required.

**License:** Code is MIT, model is OpenRAIL-M. This open-weight release ships **fixed preset voices only** (M1–M5, F1–F5) — voice cloning from a reference clip is a separate paid product called [Voice Builder](https://supertone.ai/voice-builder). For zero-shot cloning, use the Qwen3-TTS or Higgs-Audio notebooks in this repo.

In [ ]:
# @title STEP 1 — Install Dependencies
# @markdown Installs the `supertonic` SDK (4 deps: onnxruntime, numpy, soundfile, huggingface-hub) and Gradio. No GPU required.

import os, sys, subprocess
import torch  # used only for runtime info; not used by Supertonic

runtime = 'GPU' if torch.cuda.is_available() else 'CPU'
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'[Runtime] {runtime} — {gpu_name} ({vram_gb:.0f} GB)')
    print('[Note] Supertonic runs natively on ONNX CPU. A GPU offers no speedup here — you can switch to a CPU runtime if you want to free up GPU quota.')
else:
    print(f'[Runtime] {runtime} — no GPU detected. Perfect for Supertonic, which is CPU-optimized.')

print('[pip] Installing supertonic ...')
subprocess.run(['pip', 'install', '-q', '-U', 'supertonic'], check=True)
print(f'  supertonic {__import__("supertonic").__version__}')

print('[pip] Installing Gradio for the UI ...')
subprocess.run(['pip', 'install', '-q', '-U', 'gradio==5.33.0', 'tqdm==4.66.5'], check=True)

os.makedirs('/content/audio_out', exist_ok=True)
print('[Setup] /content/audio_out ready.')

In [ ]:
# @title STEP 2 — Pre-cache Models
# @markdown Downloads the ONNX models and all 10 voice styles (~400 MB) to ~/.cache/supertonic3/. Skip if you want them to download on first use.

# @markdown Reuses the Drive-cached weights from `TTS_Model_Loader.ipynb` if present,
# @markdown otherwise downloads to the default ~/.cache/huggingface (in-session only).
import pathlib
try:
    from google.colab import drive
    if not pathlib.Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive', force_remount=False)
    CACHE_ROOT = pathlib.Path('/content/drive/MyDrive/AEI_TTS_Cache')
    CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    os.environ.setdefault('HF_HOME', str(CACHE_ROOT / 'huggingface'))
    print(f'[Cache] using Drive at {CACHE_ROOT}')
except Exception:
    print('[Cache] Drive not available, using default ~/.cache/huggingface')


import os
from huggingface_hub import snapshot_download

REPO = 'Supertone/supertonic-3'
print(f'[Download] {REPO} (\u2248400 MB, ONNX + 10 voice styles) ...')
snapshot_download(REPO)
print('[Download] All assets cached.')

In [ ]:
# @title STEP 3 — Core Functions
# @markdown Wraps the Supertonic SDK. Model loads once on first use, then stays in memory.

import os, time
import numpy as np
from supertonic import TTS

VOICES = ['M1', 'M2', 'M3', 'M4', 'M5', 'F1', 'F2', 'F3', 'F4', 'F5']
LANGUAGES = [
    'en', 'ko', 'ja', 'ar', 'bg', 'cs', 'da', 'de', 'el', 'es',
    'et', 'fi', 'fr', 'hi', 'hr', 'hu', 'id', 'it', 'lt', 'lv',
    'nl', 'pl', 'pt', 'ro', 'ru', 'sk', 'sl', 'sv', 'tr', 'uk', 'vi', 'na',
]
SAMPLE_RATE = 44100

_TTS = None
_STYLES = {}

def get_tts():
    global _TTS
    if _TTS is None:
        print('[Load] Supertonic TTS engine (ONNX) ...')
        t0 = time.time()
        _TTS = TTS(auto_download=False)  # already cached by Step 2
        print(f'[Load] Ready in {time.time()-t0:.1f}s — 44.1 kHz, 99M params')
    return _TTS

def get_style(voice: str):
    if voice not in _STYLES:
        _STYLES[voice] = get_tts().get_voice_style(voice_name=voice)
    return _STYLES[voice]

def synth(text: str,
          voice: str = 'M1',
          lang: str = 'en',
          total_steps: int = 8,
          speed: float = 1.05,
          out_name: str = 'supertonic.wav') -> tuple[str, int]:
    text = (text or '').strip()
    if not text:
        raise RuntimeError('Text is empty.')
    if voice not in VOICES:
        raise RuntimeError(f'Unknown voice "{voice}". Choose from {VOICES}.')
    style = get_style(voice)
    wav, duration = get_tts().synthesize(
        text=text,
        voice_style=style,
        lang=lang,
        total_steps=int(total_steps),
        speed=float(speed),
    )
    out_path = os.path.join('/content/audio_out', out_name)
    get_tts().save_audio(wav, out_path)
    return out_path, SAMPLE_RATE, float(duration.squeeze() if hasattr(duration, 'squeeze') else duration[0])

print('[Core] Ready. Use Step 4 for the UI, Step 6 for quick test, Step 7 for batch.')

In [ ]:
# @title STEP 4 — Gradio UI
# @markdown Interactive web app. Pick a voice, pick a language, type text, hit Generate. Click the `.gradio.live` link to open.

import sys
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass
import gradio as gr

EXPRESSION_TAGS_HELP = """\
**Expression tags** (drop into your text, no extra setup):

`<laugh>` \u00b7 `<breath>` \u00b7 `<sigh>` \u00b7 `<cough>` \u00b7 `<sniff>` \u00b7 `<clear_throat>` \u00b7 `<groan>` \u00b7 `<shush>` \u00b7 `<yawn>` \u00b7 `<hum>`
"""

def _err(msg):
    return None, f'\u274c {msg}'

def ui_synth(text, voice, lang, total_steps, speed, progress=gr.Progress()):
    if not text or not text.strip():
        return _err('Type some text first.')
    try:
        progress(0.1, desc='Loading engine...')
        get_tts()
        progress(0.3, desc='Synthesizing...')
        path, sr, dur = synth(
            text.strip(),
            voice=voice,
            lang=lang,
            total_steps=int(total_steps),
            speed=float(speed),
            out_name='ui.wav',
        )
        progress(1.0, desc='Done.')
        return path, f'\u2705 {dur:.2f}s @ {sr} Hz'
    except Exception as e:
        return None, f'\u274c {e}'

with gr.Blocks(title='Supertonic 3', theme=gr.themes.Soft()) as demo:
    gr.Markdown(f'# \u26a1 Supertonic 3\n**Model:** `Supertone/supertonic-3` (99M, ONNX) \u00b7 **Runtime:** {runtime}\n\nLightning-fast on-device multilingual TTS. 31 languages, 10 voices, 44.1 kHz, expression tags.')
    with gr.Row():
        with gr.Column():
            text = gr.Textbox(
                label='Text to synthesize',
                value='The startup secured $5.2M in venture capital, a huge leap from their initial $450K seed round.',
                lines=4,
                placeholder='Type what you want the voice to say...',
            )
            with gr.Row():
                voice = gr.Dropdown(VOICES, value='M1', label='Voice')
                lang = gr.Dropdown(LANGUAGES, value='en', label='Language (or "na" for language-agnostic)')
            with gr.Accordion('Advanced', open=False):
                total_steps = gr.Slider(2, 12, value=8, step=1, label='Total steps (quality: 5 low, 8 medium, 12 high)', info="Supertonic denoising steps. 5 = low quality, 8 = medium, 12 = high.")
                speed = gr.Slider(0.7, 2.0, value=1.05, step=0.05, label='Speed (0.7x slow, 2.0x fast)', info="Playback speed factor. 1.0 = normal, 0.7 = 30% slower, 2.0 = 2x faster.")
            btn = gr.Button('Generate', variant='primary', size='lg')
        with gr.Column():
            output_audio = gr.Audio(label='Generated speech', type='filepath')
            status = gr.Markdown('')
    gr.Examples(
        examples=[
            ['The quick brown fox jumps over the lazy dog.', 'M1', 'en'],
            ['She sells seashells by the seashore.', 'F1', 'en'],
            ['The train delay was announced at 4:45 PM on Wed, Apr 3, 2024.', 'M2', 'en'],
            ['You can reach the hotel front desk at (212) 555-0142 ext. 402 anytime.', 'F2', 'en'],
            ['<laugh> That was a great joke. <breath> Anyway, as I was saying...', 'M3', 'en'],
            ['\ud56d\uc0c1 \ubc14\ub78c\uc5d0 \ub300\ud55c \ub300\ub2e8\uc744 \uc9c0\uae08 \uc2dc\uc791\ud569\ub2c8\ub2e4.', 'F3', 'ko'],
            ['La reuni\u00f3n comienza pronto y todos se sientan en silencio para escuchar.', 'F1', 'es'],
        ],
        inputs=[text, voice, lang],
    )
    gr.Markdown(EXPRESSION_TAGS_HELP)
    btn.click(
        ui_synth,
        inputs=[text, voice, lang, total_steps, speed],
        outputs=[output_audio, status],
    )
    gr.Markdown('**Tips:** The `na` language code is a language-agnostic fallback — use it for text outside the 31 supported languages. Higher `total_steps` = better quality but slower. Try expression tags for natural-sounding laughs, breaths, and sighs.')

from IPython.display import clear_output as _clear
_clear()
demo.queue(max_size=8, concurrency_limit=2).launch(
    share=True, show_error=True, server_name="0.0.0.0", server_port=7860,
)
demo.load(lambda: "🎙️ Supertonic-3 ready (CPU-only, fast). Pick a voice, choose a language, hit Generate.", outputs=[status])


In [ ]:
# @title Step 5 — Keep Alive

# @markdown Prevents Colab from disconnecting during long generation sessions. Run anytime after launching the UI.



import threading, time

def _keep():

    while True:

        time.sleep(60)

        try:

            import requests

            requests.get('https://www.google.com', timeout=5)

        except Exception:

            pass

threading.Thread(target=_keep, daemon=True).start()

print('[Keep-Alive] Running. Stop cell to disable.')

In [ ]:
# @title Step 6 — Quick Test (one-off inference, no UI)

# @markdown Run a single TTS call from the notebook. Useful for scripting or quick checks.



TEXT = "Supertonic is a lightning fast, on-device TTS system." # @param {type:"string"}

VOICE = "M1" # @param ["M1", "M2", "M3", "M4", "M5", "F1", "F2", "F3", "F4", "F5"]

LANG = "en" # @param ["en", "ko", "ja", "ar", "bg", "cs", "da", "de", "el", "es", "et", "fi", "fr", "hi", "hr", "hu", "id", "it", "lt", "lv", "nl", "pl", "pt", "ro", "ru", "sk", "sl", "sv", "tr", "uk", "vi", "na"]

TOTAL_STEPS = 8 # @param {type:"integer"}

SPEED = 1.05 # @param {type:"number"}



path, sr, dur = synth(TEXT, voice=VOICE, lang=LANG, total_steps=TOTAL_STEPS, speed=SPEED)

print(f'[Done] {path} ({dur:.2f}s @ {sr} Hz)')

from IPython.display import Audio, display

display(Audio(path))

In [ ]:
# @title Step 7 — Batch Synthesis
# @markdown Generate audio for a list of texts. Each line in `BATCH` becomes one output file.

BATCH_VOICE = 'M1' # @param ["M1", "M2", "M3", "M4", "M5", "F1", "F2", "F3", "F4", "F5"]
BATCH_LANG = 'en' # @param ["en", "ko", "ja", "ar", "bg", "cs", "da", "de", "el", "es", "et", "fi", "fr", "hi", "hr", "hu", "id", "it", "lt", "lv", "nl", "pl", "pt", "ro", "ru", "sk", "sl", "sv", "tr", "uk", "vi", "na"]
BATCH_TOTAL_STEPS = 8 # @param {type:"integer"}
BATCH_SPEED = 1.05 # @param {type:"number"}

BATCH = """\
The quick brown fox jumps over the lazy dog.
She sells seashells by the seashore.
To be, or not to be: that is the question.
It was the best of times, it was the worst of times.
All happy families are alike; each unhappy family is unhappy in its own way.
"""

lines = [l.strip() for l in BATCH.strip().splitlines() if l.strip()]
out_dir = '/content/audio_out/batch'
os.makedirs(out_dir, exist_ok=True)

for i, line in enumerate(lines):
    try:
        print(f'[{i+1}/{len(lines)}] {line[:60]}{"..." if len(line) > 60 else ""}')
        path, sr, dur = synth(
            line,
            voice=BATCH_VOICE,
            lang=BATCH_LANG,
            total_steps=BATCH_TOTAL_STEPS,
            speed=BATCH_SPEED,
            out_name=f'{i:03d}.wav',
        )
        os.rename(path, os.path.join(out_dir, f'{i:03d}.wav'))
        print(f'   \u2192 {dur:.2f}s')

    except Exception as e:
        print(f"  -> EXCEPTION: {e}")
        continue
print(f'\n[Done] {len(lines)} files written to {out_dir}')